# EdgarTools: Interactive SEC Filing Analysis

**Chapter 4: Fundamental and Alternative Data**
**Docker image**: `ml4t`
**Section Reference**: Section 4.2 (Entity Resolution and Mapping)

## Purpose

This notebook demonstrates EdgarTools, a high-level Python library for interactive
SEC EDGAR analysis. EdgarTools excels at company exploration, financial statement
extraction, and working with structured filing data like Form 4 and 13F.

## Learning Objectives

After completing this notebook, you will be able to:
- Look up companies by ticker, CIK, or search
- Retrieve and filter SEC filings
- Extract XBRL-parsed financial statements (income, balance sheet, cash flow)
- Analyze Form 4 insider transactions
- Examine 13F institutional holdings

## Cross-References

- **Related**: [`03_sec_form4_insider_transactions`](03_sec_form4_insider_transactions.ipynb) (Form 4 XML parsing)
- **Downstream**: [`04_sec_xbrl_fundamentals`](04_sec_xbrl_fundamentals.ipynb) (cross-sectional XBRL via API)

## When to Use EdgarTools

| Use Case | EdgarTools Fit |
|----------|----------------|
| Single company exploration | Excellent |
| Financial statement extraction | Excellent |
| Form 4/13F structured data | Excellent |
| Bulk downloads (1000s of filings) | Use sec-edgar instead |
| Cross-sectional fundamentals | Use XBRL Frames API |

## WARNING: Point-in-Time (PIT) Note

This notebook demonstrates **data exploration**, not PIT-correct data pipelines.
For backtesting or ML training, always use the **filing_date** (when data became
public), not the **period_end** (accounting date). Filing dates are typically
30-90 days after period end.

In [1]:
"""EdgarTools: Interactive SEC Filing Analysis — explore SEC filings, financial statements, and insider transactions."""

import os
import warnings

warnings.filterwarnings("ignore")


import polars as pl

In [2]:
# Production defaults — Papermill injects overrides for CI

In [3]:
from edgar import Company, find, get_filings, set_identity

# SEC requires a real User-Agent (name + email) for every request and blocks
# placeholder addresses. Set `EDGAR_IDENTITY` in your environment, e.g.
# `export EDGAR_IDENTITY="Jane Doe jane@example.org"`.
edgar_identity = os.environ.get("EDGAR_IDENTITY")
if not edgar_identity:
    raise RuntimeError(
        "EDGAR_IDENTITY environment variable is not set. The SEC requires a "
        "real User-Agent (name + email) for every EDGAR request and blocks "
        "placeholder addresses. Set it before running this notebook, e.g. "
        '`export EDGAR_IDENTITY="Jane Doe jane@example.org"`.'
    )
set_identity(edgar_identity)

print("EdgarTools loaded.")

EdgarTools loaded.


---
## Part 1: Company Lookup

EdgarTools provides multiple ways to find companies:
- By ticker symbol (most common)
- By CIK (SEC's Central Index Key)
- By search query

In [4]:
# Look up by ticker
apple = Company("AAPL")

print(f"=== {apple.name} ===")
print(f"CIK: {apple.cik}")
print(f"Tickers: {apple.tickers}")
print(f"SIC Code: {apple.sic}")

=== Apple Inc. ===
CIK: 320193
Tickers: ['AAPL']
SIC Code: 3571


In [5]:
# Look up by CIK (with or without zero-padding)
tesla_by_cik = Company("1318605")
print(f"Tesla by CIK: {tesla_by_cik.name}")

# Zero-padded also works
tesla_padded = Company("0001318605")
print(f"Tesla padded: {tesla_padded.name}")

Tesla by CIK: Tesla, Inc.


Tesla padded: Tesla, Inc.


In [6]:
# Search for companies by name
results = find("Microsoft")
results

    Search results for 'Microsoft'     
                                       
      Ticker   Name             Score  
 ───────────────────────────────────── 
  0     MSFT   MICROSOFT CORP   100%   
                                       

---
## Part 2: Retrieving Filings

The `get_filings()` method returns a filterable collection of filings.
Filter by form type, date range, or both.

In [7]:
# Get all filings for Apple
all_filings = apple.get_filings()
print(f"Total Apple filings: {len(all_filings)}")

Total Apple filings: 2231


In [8]:
# Filter by form type
annual_reports = apple.get_filings(form="10-K")
quarterly_reports = apple.get_filings(form="10-Q")
eightks = apple.get_filings(form="8-K")

print(f"10-K filings: {len(annual_reports)}")
print(f"10-Q filings: {len(quarterly_reports)}")
print(f"8-K filings: {len(eightks)}")

10-K filings: 32
10-Q filings: 99
8-K filings: 234


In [9]:
# Get the latest filing
latest_10k = annual_reports.latest()

print("=== Latest 10-K ===")
print(f"Filing Date: {latest_10k.filing_date}")
print(f"Accession Number: {latest_10k.accession_no}")
print(f"Is XBRL: {latest_10k.is_xbrl}")

=== Latest 10-K ===
Filing Date: 2025-10-31
Accession Number: 0000320193-25-000079
Is XBRL: 1


In [10]:
# Multiple form types at once
insider_forms = apple.get_filings(form=["3", "4", "5"])
print(f"Insider filings (Forms 3, 4, 5): {len(insider_forms)}")

Insider filings (Forms 3, 4, 5): 1378


---
## Part 3: Financial Statements from XBRL

EdgarTools parses XBRL data to provide direct access to financial statements.
This is the key differentiator from other SEC libraries.

In [11]:
# Get financials from the company (uses latest 10-K)
financials = apple.get_financials()
financials

╭──────────────────────────────────── 10-K Apple Inc. (AAPL) • CIK 0000320193 ────────────────────────────────────╮
│   Fiscal Period       Fiscal Year 2025 (ended Sep 27, 2025)                                                     │
│   Data                1,131 facts • 182 contexts                                                                │
│                                                                                                                 │
│ Periods                                                                                                         │
│   Annual              FY 2025, FY 2024, FY 2023                                                                 │
│   (1 future period after 2025-09-27 excluded)                                                                   │
│                                                                                                                 │
│ Statements (72)                                                       

### 3.1 Income Statement

In [12]:
# Get income statement
income = financials.income_statement()
income

                                                                                                     
                                  APPLE INC.   AAPL                                                  
                                  CONSOLIDATED STATEMENT OF INCOME                                   
                                  Sep 30, 2023 to Sep 27, 2025                                       
                                                                                                     
                                                       Sep 27, 2025   Sep 28, 2024   Sep 30, 2023    
   ───────────────────────────────────────────────────────────────────────────────────────────────   
            Net sales:                                     $416,161       $391,035       $383,285    
            Products                                       $307,003       $294,866       $298,085    
            Services                                       $109,158        $96,169

In [13]:
# Convert to DataFrame for analysis (wide format: one column per period)
income_df = income.to_dataframe()
date_cols = [c for c in income_df.columns if c[0].isdigit()]

print(f"Shape: {income_df.shape}")
print(f"Period columns: {date_cols}")
income_df[["label", *date_cols]].head(15)

Shape: (47, 19)
Period columns: ['2025-09-27', '2024-09-28', '2023-09-30']


,label,2025-09-27,2024-09-28,2023-09-30
0,Net sales,4.161610e+11,3.910350e+11,3.832850e+11
1,Products,3.070030e+11,2.948660e+11,2.980850e+11
2,iPhone,2.095860e+11,2.011830e+11,2.005830e+11
3,Mac,3.370800e+10,2.998400e+10,2.935700e+10
4,iPad,2.802300e+10,2.669400e+10,2.830000e+10
5,"Wearables, Home and Accessories",3.568600e+10,3.700500e+10,3.984500e+10
6,Services,1.091580e+11,9.616900e+10,8.520000e+10
7,Operating segments - Americas,1.783530e+11,1.670450e+11,1.625600e+11
8,Operating segments - Europe,1.110320e+11,1.013280e+11,9.429400e+10
9,Operating segments - Greater China,6.437700e+10,6.695200e+10,7.255900e+10


### 3.2 Balance Sheet

In [14]:
# Get balance sheet
balance = financials.balance_sheet()
balance

                                                                                                                   
                                           APPLE INC.   AAPL                                                       
                                           CONSOLIDATED BALANCE SHEETS                                             
                                           Sep 28, 2024 to Sep 27, 2025                                            
                                                                                                                   
                                                                                    Sep 27, 2025   Sep 28, 2024    
   ─────────────────────────────────────────────────────────────────────────────────────────────────────────────   
    ASSETS:                                                                                                        
      Current assets:                                                   

In [15]:
balance_df = balance.to_dataframe()
print(f"Balance sheet rows: {len(balance_df)}")

Balance sheet rows: 79


### 3.3 Cash Flow Statement

In [16]:
# Get cash flow statement
cashflow = financials.cashflow_statement()
cashflow

                                                                                                                   
                                       APPLE INC.   AAPL                                                           
                                       CONSOLIDATED STATEMENT OF CASH FLOWS                                        
                                       Sep 30, 2023 to Sep 27, 2025                                                
                                                                                                                   
                                                                     Sep 27, 2025   Sep 28, 2024   Sep 30, 2023    
   ─────────────────────────────────────────────────────────────────────────────────────────────────────────────   
      Cash, cash equivalents, and restricted cash and cash                                                         
    equivalents, ending balances                                        

### 3.4 Working with Statement Data

The DataFrame format allows you to compute ratios and perform analysis.

In [17]:
# Extract specific line items from income statement
income_df = income.to_dataframe()
date_cols = [c for c in income_df.columns if c[0].isdigit()]

key_items = income_df[income_df["label"].str.contains("Revenue|Net income", case=False, na=False)]
key_items[["label", *date_cols]]

,label,2025-09-27,2024-09-28,2023-09-30
39,Net income,1.120100e+11,9.373600e+10,9.699500e+10


---
## Part 4: Form 4 Insider Transactions

Form 4 reports insider trading activity. EdgarTools parses the XML
into structured data.

In [18]:
# Get Tesla's Form 4 filings
tesla = Company("TSLA")
form4_filings = tesla.get_filings(form="4")
print(f"Tesla Form 4 filings: {len(form4_filings)}")

Tesla Form 4 filings: 800


In [19]:
# Parse the 10 most recent Form 4 filings into a tidy DataFrame.
# Some Form 4 XML is malformed at the source; capture parse failures rather than abort.
recent_form4s = form4_filings.head(10)

records = []
for filing in recent_form4s:
    try:
        form4 = filing.obj()
        records.append(
            {
                "filing_date": str(filing.filing_date),
                "insider": form4.insider_name or "Unknown",
                "n_purchases": len(form4.common_stock_purchases),
                "n_sales": len(form4.common_stock_sales),
                "parse_error": None,
            }
        )
    except Exception as exc:
        records.append(
            {
                "filing_date": str(filing.filing_date),
                "insider": None,
                "n_purchases": None,
                "n_sales": None,
                "parse_error": str(exc)[:80],
            }
        )

insider_summary = pl.DataFrame(records)
insider_summary

filing_date,insider,n_purchases,n_sales,parse_error
str,str,i64,i64,null
"""2026-05-15""","""Vaibhav Taneja""",0,1,null
"""2026-05-04""","""Kathleen Wilson-Thompson""",0,16,null
"""2026-04-23""","""Elon Musk""",0,0,null
"""2026-04-02""","""Xiaotong Zhu""",0,0,null
"""2026-04-01""","""Kathleen Wilson-Thompson""",0,15,null
"""2026-03-09""","""Vaibhav Taneja""",0,1,null
"""2026-02-27""","""Kathleen Wilson-Thompson""",0,7,null
"""2026-01-12""","""Xiaotong Zhu""",0,0,null
"""2026-01-06""","""James R Murdoch""",0,22,null


In [20]:
# Detailed look at the first Form 4 in the window that contains stock sales.
# Wrap parsing in a guard so malformed XML doesn't abort the cell, and handle
# the case where no Form 4 in the recent window contains sales.
def _has_sales(f):
    try:
        return len(f.obj().common_stock_sales) > 0
    except Exception:
        return False


sale_filing = next((f for f in recent_form4s if _has_sales(f)), None)
if sale_filing is None:
    print("No Form 4 with stock sales found in the recent window.")
    form4 = None
else:
    form4 = sale_filing.obj()
    print(f"{form4.insider_name} — {sale_filing.filing_date}")
    print(f"Issuer: {form4.issuer}")
form4.common_stock_sales if form4 is not None else None

Vaibhav Taneja — 2026-05-15
Issuer: Issuer(cik='0001318605', name=Tesla, Inc., ticker=TSLA)


,Security,Date,Shares,Remaining,Price,AcquiredDisposed,DirectIndirect,NatureOfOwnership,form,Code,EquitySwap,footnotes,TransactionType
2,Common Stock,2026-05-13,3000,18106.5,450.0,D,D,None,4,S,False,F1,Sale


### 4.1 Form 4 Transaction Codes

| Code | Description |
|------|-------------|
| P | Open market purchase |
| S | Open market sale |
| A | Grant or award |
| M | Exercise of options |
| G | Gift |
| D | Disposition to issuer |
| F | Tax payment via shares |

---
## Part 5: 13F Institutional Holdings

Form 13F-HR reports quarterly holdings for institutional investment managers
with $100M+ in qualifying securities.

In [21]:
# Get Berkshire Hathaway's 13F filings
berkshire = Company("BRK-A")
thirteenf_filings = berkshire.get_filings(form="13F-HR")
print(f"Berkshire 13F-HR filings: {len(thirteenf_filings)}")

Berkshire 13F-HR filings: 210


In [22]:
# Get latest 13F
latest_13f = thirteenf_filings.latest()
holdings = latest_13f.obj()

print("=== Berkshire Hathaway Holdings ===")
print(f"Report Period: {holdings.report_period}")
print(f"Total Value: ${holdings.total_value:,.0f}")
print(f"Number of Holdings: {holdings.total_holdings}")

=== Berkshire Hathaway Holdings ===
Report Period: 2026-03-31
Total Value: $263,095,703,570
Number of Holdings: 90


In [23]:
# View holdings DataFrame — top 20 by reported market value.
holdings_df = holdings.holdings
top_holdings = holdings_df.nlargest(20, "Value")[["Issuer", "Class", "Value", "SharesPrnAmount"]]
top_holdings

,Issuer,Class,Value,SharesPrnAmount
0,APPLE INC,COM,57843260493,227917808
1,AMERICAN EXPRESS CO,COM,45859204536,151610700
2,COCA COLA CO,COM,30420000000,400000000
3,BANK AMERICA CORP,COM,25039178044,513624165
4,CHEVRON CORPORATION,COM,17457364606,84375856
5,OCCIDENTAL PETE CORP,COM,17221193015,264941431
6,ALPHABET INC,CAP STK CL A,15600071913,54249798
7,CHUBB LTD SWITZ,COM,11162836215,34249183
8,MOODYS CORP,COM,10762190653,24669778
9,KRAFT HEINZ CO,COM,7323527057,325634818


In [24]:
# Express each position as a share of total portfolio value.
holdings_pl = pl.from_pandas(holdings_df).with_columns(
    (pl.col("Value") / pl.col("Value").sum() * 100).alias("pct_portfolio")
)
holdings_pl.select(["Issuer", "Value", "pct_portfolio"]).head(10)

Issuer,Value,pct_portfolio
str,i64,f64
"""APPLE INC""",57843260493,21.985635
"""AMERICAN EXPRESS CO""",45859204536,17.430617
"""COCA COLA CO""",30420000000,11.562332
"""BANK AMERICA CORP""",25039178044,9.517137
"""CHEVRON CORPORATION""",17457364606,6.635367
"""OCCIDENTAL PETE CORP""",17221193015,6.5456
"""ALPHABET INC""",15600071913,5.929429
"""CHUBB LTD SWITZ""",11162836215,4.24288
"""MOODYS CORP""",10762190653,4.090599


---
## Part 6: Global Filing Search

Search across all SEC filers for specific form types.

In [25]:
# Get recent 10-K filings across all companies
recent_10ks = get_filings(form="10-K")
print(f"Recent 10-K filings available: {len(recent_10ks)}")

Recent 10-K filings available: 6189


In [26]:
# Filter by date
from datetime import date, timedelta

one_week_ago = date.today() - timedelta(days=7)
recent = get_filings(form="10-K", filing_date=f"{one_week_ago}:")
print(f"10-K filings in last week: {len(recent)}")

# Show a few
for filing in recent.head(5):
    print(f"{filing.filing_date} | {filing.company[:40]}")

10-K filings in last week: 27
2026-05-15 | AiXin Life International, Inc.
2026-05-15 | Dankon Corp
2026-05-15 | Lodging Fund REIT III, Inc.
2026-05-15 | RBC Bearings INC
2026-05-15 | Picard Medical, Inc.


---
## Part 7: Accessing Filing Content

Beyond structured data, you can access the raw filing content.

In [27]:
# Get filing content
filing = apple.get_filings(form="10-K").latest()

# Available content methods
print("=== Filing Content Methods ===")
print("filing.text()     - Plain text content")
print("filing.html()     - HTML content")
print("filing.open()     - Open in browser")
print("filing.attachments - List of attachments")

=== Filing Content Methods ===
filing.text()     - Plain text content
filing.html()     - HTML content
filing.open()     - Open in browser
filing.attachments - List of attachments


In [28]:
# Get text content (excerpt)
text_content = filing.text()
print(f"\n=== 10-K Text Excerpt ({len(text_content):,} chars total) ===")
print(text_content[:1000])


=== 10-K Text Excerpt (261,019 chars total) ===
UNITED STATES

SECURITIES AND EXCHANGE COMMISSION

Washington, D. C. 20549

FORM 10-K

(Mark One)

☒ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934

For the fiscal year ended September 27, 2025

or

☐TRANSITION REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934

For the transition period from to.

Commission File Number: 001-36743

Apple Inc.

(Exact name of Registrant as specified in its charter)

  California                                    94-2404110                              
  (State or other jurisdiction                  (I. R. S. Employer Identification No.)  
  One Apple Park Way                                                                    
  Cupertino, California                         95014                                   
  (Address of principal executive offices)      (Zip Code)                              

( 408 996-1010

(Registrant’s telephone

In [29]:
# List attachments
print("\n=== Filing Attachments ===")
for i, attachment in enumerate(filing.attachments):
    if i >= 10:
        break
    print(f"{i + 1}. {attachment.document}")


=== Filing Attachments ===
1. aapl-20250927.htm
2. a10-kexhibit4109272025.htm
3. a10-kexhibit21109272025.htm
4. a10-kexhibit23109272025.htm
5. a10-kexhibit31109272025.htm
6. a10-kexhibit31209272025.htm
7. a10-kexhibit32109272025.htm
8. aapl-20250927.xsd
9. aapl-20250927_cal.xml
10. aapl-20250927_def.xml


---
## Key Takeaways

1. EdgarTools wraps the SEC EDGAR REST API with a Pythonic surface: companies, filings, parsed XBRL statements, and structured Form 4 / 13F objects.
2. The library is well suited to interactive single-company workflows — a 10-K returns three years of income statement, balance sheet, and cash flow as ready-to-analyse DataFrames.
3. Form 4 and 13F filings expose `common_stock_purchases`, `common_stock_sales`, and a `holdings` DataFrame that fits naturally into a portfolio-concentration analysis (Berkshire's top three positions — Apple, American Express, Coca-Cola — make up about 51% of reported value in this snapshot, and the top four exceed 60%).
4. For Form 4 insider-transaction XML parsing, see [`03_sec_form4_insider_transactions`](03_sec_form4_insider_transactions.ipynb); for cross-sectional XBRL fundamentals, use the Frames API ([`04_sec_xbrl_fundamentals`](04_sec_xbrl_fundamentals.ipynb)).
5. **PIT reminder**: anything you build for backtesting must key off `filing_date`, not `period_end`. EdgarTools surfaces both — use the former for the as-of timestamp.